In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import grangercausalitytests
import warnings

# Ignore all warnings
warnings.filterwarnings("ignore")

# Load the dataset
try:
    df = pd.read_csv('DC_Master_ARIMA_Filled_pct_change_dummies.csv', parse_dates=['Date'], index_col='Date')
    df = df[df.index <= '2024-10-01']
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print("Error: 'DC_Master_Levels_and_Stationary.csv' not found.")
    # Exit or handle error appropriately if file not found
    exit()
except Exception as e:
    print(f"Error loading dataset: {e}")
    # Exit or handle other loading errors
    exit()

stationary_cols = [
    'Unemployment_Rate', # Already I(0)
    'Interest_Rate_pct_change',
    'Mortgage_Rate_pct_change',
    'CPI_pct_change',
    'House_Index_pct_change',
    'Poverty_Rate_pct_change',
    'Median_Household_Income_pct_change',
    'GDP_pct_change',
    'Population_pct_change'
]

# Verify columns exist in the DataFrame
missing_cols = [col for col in stationary_cols if col not in df.columns]
if missing_cols:
    print(f"Error: The following expected stationary columns are missing: {missing_cols}")
    # Filter out missing columns to proceed or exit
    stationary_cols = [col for col in stationary_cols if col in df.columns]
    if not stationary_cols:
        print("No stationary columns found to perform tests. Exiting.")
        exit()
    else:
        print(f"Proceeding with available columns: {stationary_cols}")


# Select only the stationary columns
df_stationary = df[stationary_cols].copy()

# --- Handle Missing Values ---
# Granger causality tests require non-missing values.
# Differencing creates NaNs at the beginning. Forecasts might have NaNs at the end.
initial_rows = len(df_stationary)
df_stationary.dropna(inplace=True)
final_rows = len(df_stationary)
print(f"Removed {initial_rows - final_rows} rows with NaN values.")

if final_rows < 10: # Need sufficient data points for the test
     print(f"Warning: Only {final_rows} complete observations available after handling NaNs. Results might be unreliable.")
     if final_rows == 0:
         print("Error: No complete observations left after removing NaNs. Cannot perform Granger Causality tests.")
         exit()


# --- Perform Granger Causality Tests ---
maxlag = 8
test_results = {}
variables = df_stationary.columns

print(f"\nPerforming Granger Causality tests for maxlag={maxlag}...")

for cause_var in variables:
    for effect_var in variables:
        if cause_var == effect_var:
            continue # Skip testing variable on itself

        print(f"Testing: {cause_var} -> {effect_var}")
        try:
            # Ensure data is float64 for the test
            test_data = df_stationary[[effect_var, cause_var]].astype(np.float64)
            # Check for constant columns which cause errors in the test
            if test_data[cause_var].nunique() <= 1 or test_data[effect_var].nunique() <= 1:
                 print(f"Skipping {cause_var} -> {effect_var} due to constant data.")
                 test_results[(cause_var, effect_var)] = {'error': 'Constant data detected'}
                 continue

            gc_result = grangercausalitytests(test_data, maxlag=maxlag, verbose=False)
            test_results[(cause_var, effect_var)] = gc_result

        except Exception as e:
            print(f"Error testing {cause_var} -> {effect_var}: {e}")
            test_results[(cause_var, effect_var)] = {'error': str(e)}


# --- Present Results ---
print("\n--- Granger Causality Test Results (p-values for F-test) ---")
print(f"Significance level: 0.1 (p < 0.1 suggests Granger causality)")

results_summary = []
for (cause_var, effect_var), result_dict in test_results.items():
    if 'error' in result_dict:
        results_summary.append({
            'Cause': cause_var,
            'Effect': effect_var,
            'Lag': 'N/A',
            'P-Value': 'Error',
            'Significant (p<0.1)': 'N/A',
            'Details': result_dict['error']
        })
        continue

    min_p_value = 1.0
    significant_lags = []
    for lag in range(1, maxlag + 1):
         # Accessing the F-test p-value: result_dict[lag][0]['ssr_ftest'][1]
         try:
            p_value = result_dict[lag][0]['ssr_ftest'][1]
            min_p_value = min(min_p_value, p_value)
            if p_value < 0.1:
                significant_lags.append(lag)
         except (IndexError, KeyError, TypeError):
             # Handle cases where test might not return expected structure for a lag
             print(f"Warning: Could not retrieve p-value for lag {lag} in {cause_var} -> {effect_var}")
             continue


    results_summary.append({
        'Cause': cause_var,
        'Effect': effect_var,
        'Min P-Value': f"{min_p_value:.4f}" if min_p_value <= 1.0 else 'N/A',
        'Significant Lags (p<0.1)': ', '.join(map(str, significant_lags)) if significant_lags else 'None'
    })

# Convert summary to DataFrame for better display
results_df = pd.DataFrame(results_summary)

# Display the summary table
print("\nSummary of Granger Causality Tests:")
print(results_df.to_string())

# Optional: Display detailed results for significant relationships
print("\n--- Detailed p-values for relationships with at least one significant lag ---")
significant_found = False
for (cause_var, effect_var), result_dict in test_results.items():
    if 'error' in result_dict: continue

    is_significant = False
    p_values_str = []
    for lag in range(1, maxlag + 1):
         try:
            p_value = result_dict[lag][0]['ssr_ftest'][1]
            p_str = f"{p_value:.3f}"
            if p_value < 0.1:
                p_str += "*"
                is_significant = True
            p_values_str.append(f"Lag {lag}: {p_str}")
         except (IndexError, KeyError, TypeError):
             p_values_str.append(f"Lag {lag}: N/A")


    if is_significant:
        significant_found = True
        print(f"\n{cause_var} -> {effect_var}:")
        print(" | ".join(p_values_str))

if not significant_found:
    print("No significant Granger causality found at p < 0.1 for any lag up to", maxlag)

Dataset loaded successfully.
Removed 0 rows with NaN values.

Performing Granger Causality tests for maxlag=8...
Testing: Unemployment_Rate -> Interest_Rate_pct_change
Testing: Unemployment_Rate -> Mortgage_Rate_pct_change
Testing: Unemployment_Rate -> CPI_pct_change
Testing: Unemployment_Rate -> House_Index_pct_change
Testing: Unemployment_Rate -> Poverty_Rate_pct_change
Testing: Unemployment_Rate -> Median_Household_Income_pct_change
Testing: Unemployment_Rate -> GDP_pct_change
Testing: Unemployment_Rate -> Population_pct_change
Testing: Interest_Rate_pct_change -> Unemployment_Rate
Testing: Interest_Rate_pct_change -> Mortgage_Rate_pct_change
Testing: Interest_Rate_pct_change -> CPI_pct_change
Testing: Interest_Rate_pct_change -> House_Index_pct_change
Testing: Interest_Rate_pct_change -> Poverty_Rate_pct_change
Testing: Interest_Rate_pct_change -> Median_Household_Income_pct_change
Testing: Interest_Rate_pct_change -> GDP_pct_change
Testing: Interest_Rate_pct_change -> Population_p